# Step 18 — federation, one module at a time

Step 07 proved that pooling `N, S, Q, G` reproduces the pooled correlation matrix exactly, and named
the blocker: `rank(G) = min(n, p)`, so a Gram matrix from a panel wider than the cohort can be
inverted back toward individual rows. The release rule is **`p < n`**.

**An earlier version of this analysis concluded federation was impossible here. That was wrong.** It
tested each cohort's whole panel as a single Gram — 430 to 771 proteins against n = 86 — which
nothing in the method requires. Modules are small. Released **one at a time**, and counted in
**proteins rather than probes**, they clear the rule with room to spare:

| cohort | trait-associated modules | individually releasable |
|---|---|---|
| A | 11 | **9** — bisque4 **7**, brown4 9, ivory **12**, darkmagenta 13, violet 18, darkolivegreen 21, darkgrey 29, lightyellow 36, greenyellow 78 |
| B | 1 | 0 — `brown` is 430 proteins |
| C | 4 | **3** — mediumpurple3 **13**, darkgreen 34, greenyellow 63 |

`bisque4` is 10 probes but **7 proteins**. Counting probes inflates `p` exactly where modules are
smallest and the rule is easiest to satisfy.

## One module per run, deliberately

This notebook federates **a single named module** and releases nothing else. Nine separate Grams
totalling 222 proteins is not obviously the same disclosure as one 222-protein Gram — each block
fixes its own proteins only up to an unknown rotation, and the blocks cannot be aligned to each
other, but **that is an argument, not a citation.** It is an open privacy question, and this
notebook is built so it never has to be answered: one module, one release, one decision.

## Cohort B contributes but never originates

B's only trait-associated module is 430 proteins, so B has nothing at this size to define. It still
ships sufficient statistics for modules defined at A and C. That asymmetry is a fair outcome of
what each cohort contains, and it is stated rather than engineered away.

In [ ]:
suppressMessages(library(WGCNA)); options(stringsAsFactors=FALSE); set.seed(42)
source("../src/paths.R")
SITES <- c("A","B","C")
W  <- lapply(SITES, function(s) readRDS(art("wgcna_%s.rds", s))); names(W) <- SITES
N   <- sapply(W, function(w) nrow(w$X)); N_MIN <- min(N)
spec <- read_traits()
build_traits <- function(m, spec){ out<-data.frame(row.names=rownames(m))
  for (i in seq_len(nrow(spec))){ v<-m[[spec$source_column[i]]]
    out[[spec$name[i]]] <- switch(spec$type[i], numeric=as.numeric(as.character(v)),
      binary=as.numeric(v=="Positive"),
      ordinal={lvl<-unique(v[!is.na(v)&v!=""]); lvl<-lvl[order(as.numeric(sub("-.*","",lvl)))]
               as.integer(factor(v,levels=lvl,ordered=TRUE))}) }; out }
TR <- lapply(SITES, function(s) build_traits(W[[s]]$meta, spec)); names(TR)<-SITES

### The algebra, unchanged from step 07

`suff()` and `pooled_cor()` are step 07's, including the clamp added in step 10: rebuilding `r` from
sufficient statistics overshoots ±1 by ~1e-16 on near-perfect pairs, and WGCNA's `checkAdjMat`
rejects that.

`federate_one()` refuses any module at or above `n`. It does not trim a module to make it fit —
trimming would be choosing which proteins to drop in order to pass a privacy rule, which is the
wrong thing to optimise.

In [ ]:
suff <- function(M){ M<-as.matrix(M)
  list(N=crossprod(!is.na(M)), S=crossprod(!is.na(M), replace(M,is.na(M),0)),
       Q=crossprod(!is.na(M), replace(M,is.na(M),0)^2), G=crossprod(replace(M,is.na(M),0))) }
pooled_cor <- function(st){ n<-st$N; s<-st$S; g<-st$G; q<-st$Q
  cov <- g/n - (s*t(s))/(n*n); sv <- sqrt(q/n - (s/n)^2)
  pmin(pmax(cov/(sv*t(sv)), -1), 1) }

federate_one <- function(ORIGIN, MODULE){
  g <- colnames(W[[ORIGIN]]$X)[W[[ORIGIN]]$mods == MODULE]
  np <- n_proteins(g)
  cat(sprintf("\n=== module '%s', defined at cohort %s : %s ===\n", MODULE, ORIGIN, size_str(g)))
  if (np >= N_MIN){
    cat(sprintf("  REFUSED: %d proteins is not < n = %d. Not trimming a module to make it fit.\n",
                np, N_MIN)); return(invisible(NULL)) }
  cat(sprintf("  eligible: %d proteins < n = %d\n", np, N_MIN))
  st <- Reduce(function(a,b) Map(`+`,a,b), lapply(SITES, function(s) suff(W[[s]]$X[,g,drop=FALSE])))
  R  <- pooled_cor(st)
  Rd <- cor(do.call(rbind, lapply(SITES, function(s) W[[s]]$X[,g,drop=FALSE])))
  ex <- max(abs(R-Rd)); cat(sprintf("  exactness gate: max|federated - pooled raw| = %.1e\n", ex))
  stopifnot(ex < 1e-9)
  n_tot <- st$N[1,1]; mu <- st$S[1,]/n_tot; sdv <- sqrt(st$Q[1,]/n_tot - mu^2)
  v <- eigen(R, symmetric=TRUE)$vectors[,1]; if (sum(v) < 0) v <- -v; names(v) <- g
  score <- lapply(SITES, function(s){
    Z <- scale(W[[s]]$X[,g,drop=FALSE], center=mu, scale=sdv); as.vector(Z %*% v) })
  names(score) <- SITES
  me <- moduleEigengenes(W[[ORIGIN]]$X[,g,drop=FALSE], rep("m", length(g)))$eigengenes[[1]]
  rt <- abs(cor(score[[ORIGIN]], me))
  cat(sprintf("  round-trip vs local moduleEigengenes at %s: r = %.4f\n", ORIGIN, rt))
  coh <- sapply(SITES, function(s){ C<-cor(W[[s]]$X[,g,drop=FALSE]); mean(C[upper.tri(C)]) })
  cat("  mean within-module correlation per site: ",
      paste(sprintf("%s %.2f", SITES, coh), collapse="  "), "\n")
  hits <- do.call(rbind, lapply(SITES, function(s){
    tr <- TR[[s]][rownames(W[[s]]$X),,drop=FALSE]
    r <- cor(score[[s]], tr, use="pairwise.complete.obs")[1,]
    p <- 2*pt(-abs(r*sqrt((N[s]-2)/(1-r^2))), N[s]-2)
    q <- p.adjust(p,"BH")
    k <- which(q<0.05)
    if (!length(k)) return(NULL)
    data.frame(site=s, trait=names(r)[k], r=round(r[k],2), q=signif(q[k],2)) }))
  cat("  trait associations at FDR 5%:\n")
  if (is.null(hits)) cat("    none\n") else print(hits[order(hits$site,-abs(hits$r)),], row.names=FALSE)
  invisible(list(module=MODULE, origin=ORIGIN, proteins=g, R=R, v=v, score=score,
                 exact=ex, roundtrip=rt, coherence=coh, hits=hits))
}

### The demonstration

`ivory` and `bisque4` originate at A, `mediumpurple3` at C. `black` and B's `brown` are included to
show the refusal path — the rule has to be able to say no, or it is not a rule.

In [ ]:
DEMO <- list(c("A","ivory"), c("A","bisque4"), c("C","mediumpurple3"),
             c("A","black"), c("B","brown"))
res <- lapply(DEMO, function(x) federate_one(x[1], x[2]))
names(res) <- sapply(DEMO, function(x) paste(x, collapse="/"))
saveRDS(res[!sapply(res, is.null)], art("step18_federated_modules.rds"))

## What federation buys

**Cohort B gains an axis it could not find alone.** On its own B yields one trait-associated module
and its only signal is renal. Scored on **C's** interferon module `mediumpurple3` — 13 proteins, a
definition B had no part in making — B shows nine associations at FDR 5%, and they are the
autoantibody set: anti-RNP-68, anti-Sm, anti-Ro60, anti-Ro52, anti-La, anti-RNP-A. The module is
present in B's data and B's own discovery could not resolve it.

**But do not read those nine as federation's doing — they are mostly the protein list's.** Step 19
separates the two. Given C's protein list and scoring it with its *own* loadings, B already reaches
four associations; the federated loadings take it to nine, and that step turns out to be a
Benjamini-Hochberg boundary artifact rather than a real gain — the two scores correlate at 0.9998
and several traits cross the line with an *unchanged or smaller* effect size. What federation adds
here is exactness and a privacy boundary, not power. Step 19 measures it across all 12 eligible
modules.

**The renal axis replicates everywhere.** `bisque4`, 7 proteins, associates with uPCR and creatinine
in **all three** cohorts. It is the one result in this repository that every cohort agrees on by
every route — own discovery, projection, and now federation.

**Interferon travels, but not identically.** `mediumpurple3` (from C) reaches all three cohorts;
`ivory` (from A) reaches A and C but not B, despite the two modules overlapping heavily. Which
cohort defines the module changes where it lands.

**Round-trip is 0.9997, not 1.0000, and that is correct.** The federated loadings centre and scale
with the **pooled** per-protein mean and sd, while a local `moduleEigengenes` fit uses that cohort's
own. The small gap is the difference between the two scalings, not a refit — step 08's projection
gate returns exactly 1.0000 because it uses a single cohort's scaling throughout.

**What this does not do.** Discovery still needs the full panel and therefore one site: the module
has to exist before it can be released. This is federated **confirmation**, which is step 07's
second option — ship module definitions discovered at one site — with the arithmetic now actually
performed rather than described.